# ランダム・インパクト・ルールの再現 (OpenAlex)

Sinatra et al., *Science* 354 (2016) / Wang & Barabási *The Science of Science* (2021) のキャリア章。

キャリア20年以上の研究者について、各論文をキャリア年次 $t$ に並べ、各論文の $c_{10}$ (出版後10年以内の被引用数) を付与する。最高傑作が位置する年次 $t^*$ の分布 $P(t)$ を、**実データ**と **$c_{10}$ をシャッフルした帰無モデル**で比較する。ルールが成り立てば両者は重なる。

重い処理 (API取得・キャッシュ・$c_{10}$計算) はすべて `src/openalex_random_impact_rule.py` 側にある。このノートブックはそれを呼ぶだけ。


In [ ]:
# src/ 内の解析モジュールを読み込む。
from pathlib import Path
import importlib
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'src' / 'openalex_random_impact_rule.py').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('リポジトリ直下を特定できません')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import openalex_random_impact_rule as rir
importlib.reload(rir)  # .py を編集したら再読込

In [ ]:
# OpenAlex クライアント。mailto を渡すと polite pool でレートが緩和される。
# 結果は data/cache/openalex_random_impact_rule.json にキャッシュされ、再実行が速くなる。
client = rir.OpenAlexClient(
    mailto="r.ikura@msp-lab.org",
    cache_path=str(ROOT / 'data' / 'cache' / 'openalex_random_impact_rule.json'),
)

## 実行

パラメータはここで調整する。`n_researchers` を増やすほど $P(t)$ は滑らかになるが、その分 API コール数と時間が増える (研究者1名あたり概ね 2 + top_k コール)。


In [ ]:
results = rir.run(
    client,
    field_filter="concepts.id:C121332964",  # Physics。None で全分野
    n_researchers=500,    # 対象研究者数 (まずは100。論文の図に寄せるなら数百〜)
    candidate_limit=1400,  # 評価する候補著者の上限
    sample_works=50000,    # 候補著者を集めるためサンプリングする論文数
    # 既定: min_career_years=20, min_papers=10, c10_window=10, cutoff_year=2015, top_k=20
)
client.save_cache()
results['stats']

### 研究者ごとの内訳


In [ ]:
df = pd.DataFrame(results['records'])
cols = ['name','first_year','last_year','span','n_eligible','t_star','best_year','best_c10','rank_frac']
df[cols].sort_values('best_c10', ascending=False).head(15)

## 可視化

左: $P(t^*)$ の実データ (赤) と シャッフル帰無モデル (黒)。両者が重なれば『最高傑作の時期はランダム』を支持する。
右: 系列内での相対位置 $i^*/N$ の分布。ランダム・インパクト・ルールなら一様 (密度1)。


In [ ]:
%matplotlib inline
fig = rir.make_figure(results)
fig

## 解釈と注意

- KS統計量が小さく $P(t)$ の相関が高いほど、実データとシャッフルが一致＝ルールを支持。
- **打ち切り**: $c_{10}$ は出版後10年が必要なため、2015年以前の論文のみを適格とした(`cutoff_year`)。直近のキャリア後半は評価から外れる。
- **名寄せ**: OpenAlex の著者同定は日本人名・中国名や企業所属で不安定 (docs/notes/openalex_exploration.md 参照)。極端に長いキャリア/多作は同名異人の併合の可能性があり、結果に混入しうる。
- **サンプル**: 候補は1980–1995の物理 works から無作為抽出した著者。書籍の元データ(Web of Science の物理学者) とは母集団が異なる点に留意。
